<a href="https://colab.research.google.com/github/codey-m/prob_stats/blob/main/module1_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1 Lab: Look & Ask
## 6.3710x — Probability and Statistical Data Analysis

### The Story

In 1893, biologist W.F.R. Weldon measured over a thousand crabs from the Bay of Naples. He was interested in a simple question: **does the ratio of a crab's frontal features to its body length come from a single population — or could there be subgroups hiding in the data?**

That question sparked one of the most important debates in the history of statistics, leading Karl Pearson to develop the chi-squared test.

Today, you have measurements from 50 crabs. Two columns:
- **FL**: Frontal lobe size (mm)
- **CL**: Carapace (shell) length (mm)

Your job: **look at the data, describe what you see, and pose a question worth answering.**

### Libraries You'll Use

| Library | What it does | How you'll use it today |
|---|---|---|
| `pandas` | Data loading and manipulation | Read CSV, inspect, compute new columns |
| `numpy` | Numerical computation | Boolean operations, counting |
| `matplotlib` | Core plotting | Histograms, scatter plots |
| `seaborn` | Statistical visualization | Distribution plots with built-in KDE |


In [15]:
# ── Setup: Data Loading for Colab ──
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11

def load_data(filename):
    """Load CSV from local directory or GitHub."""
    if os.path.exists(filename):
        return pd.read_csv(filename)

    base_url = "https://raw.githubusercontent.com/codey-m/prob_stats/main/"
    url = base_url + filename
    print(f"📥 Loading from GitHub: {filename}")
    return pd.read_csv(url)

## Part 1: Load and Inspect

The first thing you do with any dataset: **look at it**. How many rows? How many columns? What are the types? What are the ranges?


In [ ]:
crabs = load_data("crabs_module1.csv")

print("Shape:", crabs.shape)
print("Columns:", list(crabs.columns))

In [ ]:
crabs.head(10)

In [ ]:
crabs.describe().round(2)

### TODO 1

Use the output of `describe()` above to answer:

- How many crabs are in the dataset?
- What is the minimum value of CL?
- What is the maximum value of FL?


In [ ]:
# TODO 1: Fill in these values from the table above
n_crabs = ...
cl_min = ...
fl_max = ...

print(f"Number of crabs: {n_crabs}")
print(f"Minimum CL: {cl_min}")
print(f"Maximum FL: {fl_max}")

## Part 2: Compute the Ratio

Weldon didn't study FL and CL separately — he studied their **ratio**. Why?

Because the ratio removes the effect of overall body size. A big crab and a small crab might have the same ratio if their proportions are similar. Dividing FL by CL isolates the *shape* of the crab from its *size*.


### TODO 2

Create a new column called `"ratio"` equal to FL divided by CL.

Then compute its mean and standard deviation.


In [ ]:
# TODO 2: Compute the ratio
crabs["ratio"] = ...  # FL divided by CL

mean_r = ...  # mean of the ratio column
std_r = ...   # standard deviation of the ratio column

print(f"Mean ratio:  {mean_r:.4f}")
print(f"Std dev:     {std_r:.4f}")

**Check yourself:** The mean should be between 0.4 and 0.6, and the standard deviation should be much smaller than the mean. If not, double-check your formula.


## Part 3: Histograms — and Why Bin Width Matters

A histogram divides the range of your data into **bins** (equal-width intervals) and counts how many observations fall in each one. It's the most common way to visualize a distribution.

But a histogram requires a choice: **how many bins?**

This isn't cosmetic — it changes what patterns you can see. Too few bins and you blur out real structure. Too many and you see noise that isn't there. Let's explore this.


In [ ]:
# Let's look at FL with three different bin counts
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, n_bins in zip(axes, [5, 15, 50]):
    ax.hist(crabs["FL"], bins=n_bins, color="steelblue",
            edgecolor="white", alpha=0.8)
    ax.set_xlabel("Frontal Lobe Size (mm)")
    ax.set_ylabel("Frequency")
    ax.set_title(f"{n_bins} bins")

plt.suptitle("Same data, different bin counts", fontsize=14,
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### TODO 3

Look at the three histograms above.

- With 5 bins, what do you see?
- With 50 bins, what do you see?
- Which bin count do you think best represents the shape of the data?

A common rule of thumb: start with approximately $\sqrt{n}$ bins, where $n$ is the number of data points.


In [ ]:
# TODO 3: Compute the square-root rule for our dataset
n = len(crabs)
sqrt_n_bins = ...  # round the square root of n to the nearest integer

print(f"n = {n}")
print(f"sqrt(n) rule suggests: {sqrt_n_bins} bins")

Now use your computed bin count to plot FL and CL side by side.


In [ ]:
# TODO 3 (continued): Use sqrt_n_bins for both histograms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(crabs["FL"], bins=sqrt_n_bins, color="steelblue",
             edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Frontal Lobe Size (mm)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of FL")

axes[1].hist(crabs["CL"], bins=sqrt_n_bins, color="coral",
             edgecolor="white", alpha=0.8)
axes[1].set_xlabel("Carapace Length (mm)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution of CL")

plt.tight_layout()
plt.show()

## Part 4: The Relationship Between FL and CL

Histograms show one variable at a time. A **scatter plot** shows two variables together — each crab becomes a point in 2D space.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(crabs["CL"], crabs["FL"], color="steelblue",
           edgecolor="white", s=80, alpha=0.7, linewidth=0.5)
ax.set_xlabel("Carapace Length (mm)")
ax.set_ylabel("Frontal Lobe Size (mm)")
ax.set_title("FL vs CL")

plt.tight_layout()
plt.show()

### TODO 4

If every crab had exactly the same FL/CL ratio, what would this scatter plot look like?

Let's test that. A constant ratio means $FL = r \cdot CL$ for some fixed value $r$. That's a straight line through the origin with slope equal to the ratio.

Note that we may continue to reference the variable `mean_r`


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(crabs["CL"], crabs["FL"], color="steelblue",
           edgecolor="white", s=80, alpha=0.7, linewidth=0.5)
ax.set_xlabel("Carapace Length (mm)")
ax.set_ylabel("Frontal Lobe Size (mm)")
ax.set_title("FL vs CL")

# TODO 4: Add a reference line using the mean ratio
cl_range = np.linspace(crabs["CL"].min(), crabs["CL"].max(), 100)
ax.plot(cl_range, ..., "--", color="red",  # what goes on the y-axis?
        alpha=0.6, linewidth=2,
        label=f"Constant ratio = {mean_r:.3f}")
ax.legend()

plt.tight_layout()
plt.show()

**What to notice:**
- The points cluster near the line but don't fall exactly on it
- Some crabs sit above (higher ratio), some below (lower ratio)
- Is the scatter *around* the line random, or is there structure?


## Part 5: The Ratio Distribution

This is the key plot. Weldon stared at a version of this for months.

We'll use `seaborn` here. Its `histplot` function combines a histogram with a smooth density estimate (called a **KDE**, or kernel density estimate) in a single call. Think of the KDE as a smoothed version of the histogram — it estimates the underlying shape of the distribution without depending on bin edges.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sns.histplot(crabs["ratio"], bins=sqrt_n_bins, kde=True,
             color="seagreen", edgecolor="white", alpha=0.7,
             linewidth=0.5, line_kws={"linewidth": 2.5}, ax=ax)

ax.set_xlabel("FL / CL Ratio")
ax.set_ylabel("Count")
ax.set_title("Distribution of FL/CL Ratio")

plt.tight_layout()
plt.show()

**Look carefully at this plot.**

- Does the smoothed curve have a single peak (unimodal)?
- Or do you see hints of a second bump (bimodal)?
- Is the distribution symmetric?

Hold that thought.


## Part 6: Empirical Proportions

One of the simplest things you can compute from data: **what fraction of observations satisfy some condition?**

This is an empirical proportion — a direct estimate of a probability. If 30 out of 50 crabs have a ratio above some threshold, our best estimate of the probability is 30/50 = 0.60.


### TODO 5

Compute the proportion of crabs whose ratio falls within one standard deviation of the mean — that is, crabs whose ratio satisfies:

$$\text{mean} - \text{std} < \text{ratio} < \text{mean} + \text{std}$$

For a bell-shaped (normal) distribution, this proportion should be approximately **0.68**. How does your data compare?

Note that we may continue to reference vars `mean_r` and `std_r`


In [ ]:
# TODO 5: Compute the proportion within 1 standard deviation

lower_bound = ...  # mean minus one standard deviation
upper_bound = ...  # mean plus one standard deviation

# Boolean array: True for each crab within the bounds
within_1sd = ...

# Proportion: use np.mean on a boolean array (True=1, False=0)
prop_within_1sd = ...

print(f"Mean:  {mean_r:.4f}")
print(f"Std:   {std_r:.4f}")
print(f"Bounds: [{lower_bound:.4f}, {upper_bound:.4f}]")
print(f"Proportion within 1 std dev: {prop_within_1sd:.2f}")
print(f"Expected for a normal distribution: 0.68")

### TODO 6

Now do the same for **two** standard deviations. For a normal distribution, approximately **0.95** of values fall within two standard deviations of the mean.


In [ ]:
# TODO 6: Proportion within 2 standard deviations
lower_2sd = ...
upper_2sd = ...
within_2sd = ...
prop_within_2sd = ...

print(f"Bounds: [{lower_2sd:.4f}, {upper_2sd:.4f}]")
print(f"Proportion within 2 std dev: {prop_within_2sd:.2f}")
print(f"Expected for a normal distribution: 0.95")

## Part 7: The Question

Let's bring it all together. Here's the ratio distribution one final time, annotated with the bounds you just computed.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.histplot(crabs["ratio"], bins=sqrt_n_bins, kde=True,
             color="seagreen", edgecolor="white", alpha=0.6,
             linewidth=0.5, line_kws={"linewidth": 3}, ax=ax)

# Mark the mean
ax.axvline(mean_r, color="red", linestyle="--", linewidth=2,
           label=f"Mean = {mean_r:.3f}")

# Shade the 1-std-dev region
ax.axvspan(lower_bound, upper_bound, alpha=0.15, color="red",
           label=f"±1 std dev ({prop_within_1sd:.0%} of data)")

# Shade the 2-std-dev region
ax.axvspan(lower_2sd, upper_2sd, alpha=0.08, color="blue",
           label=f"±2 std dev ({prop_within_2sd:.0%} of data)")

ax.set_xlabel("FL / CL Ratio")
ax.set_ylabel("Count")
ax.set_title("One Population — Or Two?",
             fontsize=14, fontweight="bold")
ax.legend()

plt.tight_layout()
plt.show()

## Weldon's Question

In 1893, W.F.R. Weldon looked at a distribution like this and wondered:

> *"Is this the natural variation within a single population? Or is this actually two overlapping groups — perhaps two subspecies — each with their own typical ratio?"*

He couldn't answer the question with the tools available at the time. He wrote to Karl Pearson, who developed new statistical methods specifically to address it.

**By the end of this course, you will be able to:**
- Model this distribution with probability distributions (Module 2)
- Measure the relationships between multiple crab features (Module 3)
- Estimate parameters and quantify your uncertainty (Module 4)
- Formally test whether this is one group or two (Module 5)

Right now, all you can do is look. And that's exactly where good statistics starts.


## Report Your Results

Enter these values on the course platform. Round as indicated.


In [ ]:
# Run this cell to generate your report values
print("="*50)
print("MODULE 1 REPORT VALUES")
print("="*50)
print(f"R1. Number of crabs:              {n_crabs}")
print(f"R2. Mean ratio (4 decimals):      {mean_r:.4f}")
print(f"R3. Std dev of ratio (4 decimals):{std_r:.4f}")
print(f"R4. sqrt(n) bin count:            {sqrt_n_bins}")
print(f"R5. Prop within 1 std dev (2 dec):{prop_within_1sd:.2f}")
print(f"R6. Prop within 2 std dev (2 dec):{prop_within_2sd:.2f}")

## Reflection Questions

Answer these below. There are no single right answers — these are about building your intuition.

**Q1.** In Part 3, you saw the same data with 5, 15, and 50 bins. Describe specifically what changed. Which features of the data were visible with few bins? Which appeared only with many bins? Which might be noise rather than real structure?


*Your answer here*

**Q2.** In Part 4, some crabs fall above the red line and some below. If the deviations from the line are purely random, would you expect any pattern in *which* crabs fall above vs below? Look at the plot — do you see a pattern?


*Your answer here*

**Q3.** Your empirical proportions from Parts 5–6 either matched or didn't match the 68/95 rule for normal distributions. What does this tell you about whether the ratio distribution is well-described by a bell curve? What else could explain a mismatch?


*Your answer here*

**Q4.** You have two measurements per crab (FL and CL). If you could add ONE more piece of information about each crab, what would you want to know, and why?


*Your answer here*